# Step 2 — Combine and Aggregate files to Processed folder

This notebook loads the latest raw run (`data/raw/run_*/`) and writes normalized outputs to `data/processed/`.

## Input (latest run folder)
- `timeseries_data.csv` — API time-series data (daily)
- `query_metadata.csv` — query metadata lookup

## Outputs (`data/processed/`)
- `query_metadata.csv` — metadata lookup table
- `all_words_daily.csv` — all queries, daily
- `chosen_words_daily.csv` — canonical only, daily
- `all_words_weekly.csv` — all queries, weekly aggregated
- `chosen_words_weekly.csv` — canonical only, weekly aggregated
- `chosen_words_weekly_pivoted.csv` — wide pivot for figures (languages × weeks)


In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

# Force config reload (Jupyter caches imports)
if 'config' in sys.modules:
    del sys.modules['config']

from config import (
    RAW_DIR, PROCESSED_DIR,
    QUERY_METADATA_FILE, ALL_WORDS_DAILY_FILE, CHOSEN_WORDS_DAILY_FILE,
    ALL_WORDS_WEEKLY_FILE, CHOSEN_WEEKLY_FILE, CHOSEN_WEEKLY_PIVOT_FILE,
    LANGUAGE_ORDER
)

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from shutil import copy2

print(f"✓ Config loaded")
print(f"  PROCESSED_DIR: {PROCESSED_DIR}")

## 2. Aggregate by week

In [ ]:
# Find the latest run folder that has timeseries_data.csv
run_dirs = sorted(RAW_DIR.glob('run_*'))
latest_run = None
for run_dir in reversed(run_dirs):
    if (run_dir / 'timeseries_data.csv').exists():
        latest_run = run_dir
        break

if latest_run is None:
    raise FileNotFoundError("No run directory with timeseries_data.csv found")

print(f'✓ Using latest run: {latest_run.name}')
print()

# Load timeseries (API data only)
ts_file = latest_run / 'timeseries_data.csv'
timeseries = pd.read_csv(ts_file)
timeseries['date'] = pd.to_datetime(timeseries['date'])
print(f'✓ Timeseries loaded: {len(timeseries):,} rows')
print(f'  Columns: {list(timeseries.columns)}')
print()

# Load metadata (lookup table - keep separate)
meta_file = latest_run / 'query_metadata.csv'
query_metadata = pd.read_csv(meta_file)
print(f'✓ Metadata loaded: {len(query_metadata):,} unique query+language pairs')
print(f'  Columns: {list(query_metadata.columns)}')
print()

timeseries.head(3)

## 3. Prepare for aggregation (copy metadata, verify timeseries)

In [ ]:
# Copy query_metadata.csv to processed/ so it's accessible with aggregated data
copy2(meta_file, QUERY_METADATA_FILE)
print(f'✓ Copied: {meta_file.name} → {PROCESSED_DIR.name}/')
print(f'  Use this to merge with timeseries data on [query, language_ISO]')
print()

## 4. Weekly aggregation (API data only)

In [ ]:
# Aggregate timeseries by week (API data only - NO metadata columns)
timeseries_weekly = (
    timeseries
    .groupby(['query', 'language_ISO', pd.Grouper(key='date', freq='W')])
    .agg(
        count       = ('count',       'sum'),
        count_no_rt = ('count_no_rt', 'sum'),
        rank        = ('rank',        'first'),
        freq        = ('freq',        'mean'),
        freq_no_rt  = ('freq_no_rt',  'mean'),
    )
    .reset_index()
)

print(f'✓ Weekly aggregated: {len(timeseries_weekly):,} rows')
print(f'  Columns: {list(timeseries_weekly.columns)}')
print()
timeseries_weekly.head(3)

## 5. Save processed outputs

## 5. Save processed outputs

### 5a. all_words_weekly.csv — all queries, weekly aggregated, API data only

In [ ]:
timeseries.to_csv(ALL_WORDS_DAILY_FILE, index=False)
size_mb = ALL_WORDS_DAILY_FILE.stat().st_size / 1024 / 1024
print(f'✓ Saved: all_words_daily.csv')
print(f'  Rows: {len(timeseries):,}')
print(f'  Size: {size_mb:.1f} MB')
print()

In [ ]:
timeseries_weekly.to_csv(ALL_WORDS_WEEKLY_FILE, index=False)
size_mb = ALL_WORDS_WEEKLY_FILE.stat().st_size / 1024 / 1024
print(f'✓ Saved: all_words_weekly.csv')
print(f'  Rows: {len(timeseries_weekly):,}')
print(f'  Size: {size_mb:.1f} MB')
print()

In [ ]:
# ============================================================================
# Save daily data (chosen words only)
# ============================================================================
chosen_daily = (
    timeseries
    .merge(
        query_metadata[['query', 'ISO', 'Canonical']].rename(columns={'ISO': 'language_ISO'}),
        on=['query', 'language_ISO'],
        how='left'
    )
    .query('Canonical == 1')
    .drop('Canonical', axis=1)
)

chosen_daily.to_csv(CHOSEN_WORDS_DAILY_FILE, index=False)
size_mb = CHOSEN_WORDS_DAILY_FILE.stat().st_size / 1024 / 1024
print(f'✓ Saved: chosen_words_daily.csv')
print(f'  Rows: {len(chosen_daily):,}')
print(f'  Size: {size_mb:.1f} MB')
print()


### 5b. chosen_words_weekly.csv — canonical words, weekly aggregated, API data only

Filtered by `Canonical == 1` in `query_metadata.csv`. Merge with metadata to add linguistic info.

In [ ]:
# Filter to canonical words only (merge just to get Canonical flag, then drop it)
canonical_weekly = (
    timeseries_weekly
    .merge(
        query_metadata[['query', 'ISO', 'Canonical']].rename(columns={'ISO': 'language_ISO'}),
        on=['query', 'language_ISO'],
        how='left'
    )
    .query('Canonical == 1')
    .drop('Canonical', axis=1)  # Remove Canonical column after filtering
)

canonical_weekly.to_csv(CHOSEN_WEEKLY_FILE, index=False)
size_mb = CHOSEN_WEEKLY_FILE.stat().st_size / 1024 / 1024
print(f'✓ Saved: {CHOSEN_WEEKLY_FILE.name}')
print(f'  Rows: {len(canonical_weekly):,}')
print(f'  Languages: {canonical_weekly["language_ISO"].nunique()}')
print(f'  Size: {size_mb:.1f} MB')
print(f'  Columns: date, query, language_ISO, count, count_no_rt, rank, freq, freq_no_rt')
print()
canonical_weekly.head(3)


### 5c. chosen_words_weekly_pivoted.csv — wide pivot for figures

In [ ]:
# Build ISO -> Language map from metadata for paper-order plotting
iso_to_language = (
    query_metadata[['ISO', 'Language']]
    .dropna()
    .drop_duplicates()
    .set_index('ISO')['Language']
    .to_dict()
)

pivot_input = (
    canonical_weekly
    .groupby(['language_ISO', 'date'])['freq_no_rt']
    .sum()
    .reset_index()
)

# Convert ISO rows to full language names so LANGUAGE_ORDER applies directly
pivot_input['language'] = pivot_input['language_ISO'].map(iso_to_language).fillna(pivot_input['language_ISO'])

pivot = pivot_input.pivot(index='language', columns='date', values='freq_no_rt')

ordered = [l for l in LANGUAGE_ORDER if l in pivot.index]
missing = [l for l in pivot.index if l not in LANGUAGE_ORDER]
if missing:
    print(f'WARNING: not in LANGUAGE_ORDER: {missing}')
pivot = pivot.reindex(ordered + missing)

pivot.to_csv(CHOSEN_WEEKLY_PIVOT_FILE)
size_mb = CHOSEN_WEEKLY_PIVOT_FILE.stat().st_size / 1024 / 1024
print(f'✓ Saved: {CHOSEN_WEEKLY_PIVOT_FILE.name}')
print(f'  Shape: {pivot.shape[0]} languages x {pivot.shape[1]} weeks')
print(f'  Size: {size_mb:.1f} MB')
print()
pivot.iloc[:3, :4]

In [ ]:
print('=' * 70)
print('OUTPUTS WRITTEN TO data/processed/')
print('=' * 70)

output_files = [
    ('query_metadata.csv', QUERY_METADATA_FILE),
    ('all_words_daily.csv', ALL_WORDS_DAILY_FILE),
    ('chosen_words_daily.csv', CHOSEN_WORDS_DAILY_FILE),
    ('all_words_weekly.csv', ALL_WORDS_WEEKLY_FILE),
    ('chosen_words_weekly.csv', CHOSEN_WEEKLY_FILE),
    ('chosen_words_weekly_pivoted.csv', CHOSEN_WEEKLY_PIVOT_FILE),
]

for name, fpath in output_files:
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1024 / 1024
        print(f'  {name:<40}  {size_mb:>7.1f} MB')

print()
print('📋 Data structure (all API data, metadata kept separate):')
print('  ')
print('  query_metadata.csv')
print('    ├─ Lookup table: [query, ISO, Language, Canonical, Type, ...]')
print('    └─ Merge with timeseries: metadata[ISO] == timeseries[language_ISO]')
print('  ')
print('  all_words_daily.csv (all queries, daily)')
print('    ├─ [date, query, language_ISO, count, count_no_rt, rank, freq, freq_no_rt]')
print('    └─ Detailed time-series, all queries')
print('  ')
print('  chosen_words_daily.csv (canonical only, daily)')
print('    ├─ [date, query, language_ISO, count, count_no_rt, rank, freq, freq_no_rt]')
print('    └─ Pre-filtered to Canonical==1')
print('  ')
print('  all_words_weekly.csv (all queries, weekly)')
print('    ├─ [date, query, language_ISO, count, count_no_rt, rank, freq, freq_no_rt]')
print('    └─ Weekly aggregated, all queries')
print('  ')
print('  chosen_words_weekly.csv (canonical only, weekly)')
print('    ├─ [date, query, language_ISO, count, count_no_rt, rank, freq, freq_no_rt]')
print('    └─ Weekly aggregated, pre-filtered to Canonical==1')
print('  ')
print('  chosen_words_weekly_pivoted.csv')
print('    ├─ Wide format: language names (rows) × weeks (columns)')
print('    └─ Ready for figure generation')
print()
print('✓ Pipeline complete!')